# Premier notebook · Le fichier de Comtoise Auto Occasion

**ELC2 · Introduction au machine learning · Séance 1**

Vous allez refaire en Python ce que nous avons fait en cours sur le fichier de Nathalie.
Vous n'avez **rien à écrire** : il suffit d'exécuter les cellules.

**Comment ça marche.** Cliquez dans une cellule grise, puis appuyez sur **Maj + Entrée**.
Le résultat s'affiche juste en dessous. Faites-le pour chaque cellule, **dans l'ordre**,
de haut en bas.

Si quelque chose part de travers : menu **Run → Run All Cells**. Tout recommence depuis le
fichier Excel, qui, lui, n'est jamais modifié.

## 0. Vérifier que vous êtes dans l'environnement du cours

Cette cellule affiche quel Python exécute votre notebook. Il doit être celui du dossier
`.venv` de votre dossier `ia-elc2`, et pas celui de votre système.

In [ ]:
import sys
from pathlib import Path

python_utilise = Path(sys.executable)
print("Python utilisé :", ".../" + "/".join(python_utilise.parts[-4:]))

if ".venv" in python_utilise.parts:
    print("OK : ce notebook tourne dans l'environnement virtuel du cours.")
else:
    print("ATTENTION : ce n'est pas le Python du cours. Relisez le guide d'installation.")

## 1. Regarder avant de toucher

`pandas` est la bibliothèque qui manipule des tableaux. On la charge sous le petit nom `pd`,
puis on ouvre le fichier de Nathalie. Le tableau s'appelle désormais `ventes`.

In [ ]:
import pandas as pd

ventes = pd.read_excel("donnees/ventes-3-agences.xlsx")
ventes.head(20)

Ce sont les 20 lignes de l'extrait imprimé dans le mail. Attention : Python compte à partir
de **0**, donc la ligne 1 du mail est la ligne `0` ici.

Les cases vides s'affichent `NaN` : c'est la façon qu'a Python d'écrire « rien ».

Combien de lignes et de colonnes en tout ?

In [ ]:
ventes.shape

## 2. Les doublons

Combien de lignes sont la copie exacte d'une ligne déjà vue ?

In [ ]:
print("Doublons :", ventes.duplicated().sum())

La vente `V-0004`, par exemple, apparaît deux fois :

In [ ]:
ventes[ventes["ref"] == "V-0004"]

On garde un seul exemplaire de chaque ligne, puis on recompte.

In [ ]:
ventes = ventes.drop_duplicates()
len(ventes)

## 3. Le texte qui dit la même chose

Combien d'écritures différentes pour nos trois agences ?

In [ ]:
ventes["agence"].value_counts()

Deux lignes semblent identiques. Affichons chaque écriture entre guillemets, pour voir où elle
commence et où elle finit :

In [ ]:
list(ventes["agence"].unique())

`'Belfort '` et `'Montbéliard '` ont un espace à la fin. Invisible dans Excel, mais pour
l'ordinateur, ce sont d'autres agences.

On corrige en trois gestes :

| Geste | Ce qu'il fait |
|---|---|
| `.str.strip()` | enlève les espaces aux deux bouts |
| `.str.lower()` | écrit tout en minuscules |
| `.map(...)` | remplace chaque écriture grâce à une table de correspondance |

In [ ]:
bonne_ecriture = {
    "montbéliard": "Montbéliard",
    "montbeliard": "Montbéliard",
    "belfort": "Belfort",
    "besançon": "Besançon",
    "besancon": "Besançon",
}

ventes["agence"] = ventes["agence"].str.strip().str.lower().map(bonne_ecriture)
ventes["agence"].value_counts()

La marque et l'état ont le même défaut, en plus petit :

In [ ]:
ventes["marque"] = ventes["marque"].str.title().replace("Citroen", "Citroën")
ventes["etat"] = ventes["etat"].replace("Tres bon", "Très bon")
ventes["etat"].value_counts()

### Le carburant

Même problème dans la colonne `carburant`. `dropna=False` affiche aussi les cases vides.

In [ ]:
ventes["carburant"].value_counts(dropna=False)

Même méthode, avec une table de correspondance. Elle contient une ligne qu'aucune règle
d'écriture ne devine : **Gazole et Gasoil, c'est du Diesel**. Il faut connaître le métier.

In [ ]:
carburant_correct = {
    "essence": "Essence",
    "diesel": "Diesel",
    "hybride": "Hybride",
    "électrique": "Électrique",
    "electrique": "Électrique",
    "elec": "Électrique",
    "gazole": "Diesel",
    "gasoil": "Diesel",
}

ventes["carburant"] = ventes["carburant"].str.strip().str.lower().map(carburant_correct)
ventes["carburant"].value_counts(dropna=False)

## 4. Les nombres écrits comme du texte

Regardez la colonne `kilometrage` à la ligne `2` : « 145 395 km ». Pour vous, c'est un
nombre. Pour l'ordinateur, c'est du texte : impossible d'en calculer une moyenne.

In [ ]:
ventes[["ref", "kilometrage", "prix"]].head(8)

On enlève les espaces, le « km » et le « € », puis on demande à pandas de lire le reste
comme un nombre.

In [ ]:
km = ventes["kilometrage"].astype("string")
km = km.str.replace(" ", "").str.replace("km", "")
ventes["kilometrage"] = pd.to_numeric(km).astype(float)

prix = ventes["prix"].astype("string")
prix = prix.str.replace(" ", "").str.replace("€", "")
ventes["prix"] = pd.to_numeric(prix).astype(float)

ventes[["ref", "kilometrage", "prix"]].head(8)

## 5. Les distributions

Maintenant que ce sont de vrais nombres, on peut regarder **comment ils se répartissent**.
`describe()` résume une colonne : combien de valeurs, la moyenne (`mean`), l'écart-type
(`std`), le minimum, les quartiles (`25%`, `50%` = la médiane, `75%`) et le maximum.

In [ ]:
ventes[["kilometrage", "remise", "prix"]].describe().round(1)

Regardez le `max` de chaque colonne : 1 250 000 km, une remise de 48 %, un prix de 77 000 €.
On y revient à la partie 6.

Un **histogramme** range les valeurs par tranches et compte combien il y en a dans chaque
tranche. C'est le dessin de la distribution.

In [ ]:
ventes["prix"].plot(kind="hist", bins=range(0, 80001, 2000), figsize=(9, 3), title="Prix (€)");

In [ ]:
ventes["remise"].plot(kind="hist", bins=40, figsize=(9, 3), title="Remise (%)");

Deux formes très différentes :

- le **prix** a une longue queue à droite : beaucoup de voitures bon marché, quelques chères ;
- la **remise** fait une cloche autour de 6 %, à part un point isolé, très loin, à droite.

Sur une distribution avec une longue queue, la moyenne est tirée vers la queue. La médiane,
non :

In [ ]:
print("Prix moyen  :", round(ventes["prix"].mean()), "€")
print("Prix médian :", ventes["prix"].median(), "€")

## 6. Les valeurs aberrantes

### Méthode 1 : le z-score, pour une distribution en cloche

Le z-score dit à combien d'**écarts-types** une valeur est de la moyenne. Au-delà de 3, c'est
très rare dans une cloche : 3 valeurs sur 1 000. On l'utilise sur la **remise**, qui a la forme
d'une cloche.

`std(ddof=0)` : on divise par le nombre de valeurs, comme au tableau.

In [ ]:
moyenne = ventes["remise"].mean()
ecart_type = ventes["remise"].std(ddof=0)
z = (ventes["remise"] - moyenne) / ecart_type

ventes[z.abs() > 3]

Une seule ligne : la remise de 48 %. La virgule de 4,8 a sauté.

### Méthode 2 : les quartiles, qui ne supposent aucune forme

On range les prix, on coupe en quatre. Entre le premier quartile (Q1) et le troisième (Q3), il
y a la moitié des voitures : c'est l'**écart interquartile** (IQR). Tout ce qui dépasse Q3 de
plus d'une fois et demie cet écart est signalé.

In [ ]:
q1 = ventes["prix"].quantile(0.25)
q3 = ventes["prix"].quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr
print("Q1 =", q1, " Q3 =", q3, " IQR =", iqr)
print("bornes :", borne_basse, "et", borne_haute)

ventes[(ventes["prix"] < borne_basse) | (ventes["prix"] > borne_haute)]

Cinq voitures signalées. Le 77 000 € est une erreur. Les quatre autres sont des voitures
**chères, mais vraies** : c'est la longue queue de la distribution. **Détecter n'est pas
supprimer** : on regarde chaque ligne avant de décider.

Et les deux prix à 1 € ? Aucune des deux méthodes ne les voit. Il faut une **borne fixée avec
le métier** : aucune voiture ne se vend moins de 1 000 €.

### La décision : on ne devine pas

On vide les cases impossibles. `mask` veut dire : « cache la valeur quand cette condition
est vraie ».

In [ ]:
ventes["annee"] = ventes["annee"].mask(ventes["annee"] > 2025)
ventes["kilometrage"] = ventes["kilometrage"].mask(ventes["kilometrage"] > 500000)
ventes["prix"] = ventes["prix"].mask((ventes["prix"] < 1000) | (ventes["prix"] > 50000))
ventes["remise"] = ventes["remise"].mask(ventes["remise"] > 30)

## 7. Les cases vides

Combien de cases vides, colonne par colonne ?

In [ ]:
ventes.isna().sum()

Une décision par colonne, comme en cours :

| Colonne | Décision | Pourquoi |
|---|---|---|
| `prix` | supprimer la ligne | c'est la cible : sans elle, la ligne ne sert à rien |
| `annee` | supprimer la ligne | une seule ligne concernée |
| `kilometrage` | remplir avec la **médiane** | la distribution a une longue queue : la moyenne serait tirée vers le haut |
| `remise` | remplir avec la médiane | une seule case |
| `carburant` | écrire « Inconnu » | on garde la ligne, sans inventer de carburant |

In [ ]:
ventes = ventes.dropna(subset=["prix", "annee"])
ventes["annee"] = ventes["annee"].astype(int)

ventes["kilometrage"] = ventes["kilometrage"].fillna(ventes["kilometrage"].median())
ventes["remise"] = ventes["remise"].fillna(ventes["remise"].median())
ventes["carburant"] = ventes["carburant"].fillna("Inconnu")

ventes.isna().sum()

### La réponse à Nathalie : combien de ventes utilisables ?

In [ ]:
len(ventes)

## 8. La corrélation : deux colonnes ensemble

`corr()` calcule le coefficient r pour chaque paire de colonnes. La diagonale vaut 1 : une
colonne est parfaitement corrélée avec elle-même.

In [ ]:
ventes[["annee", "kilometrage", "remise", "prix"]].corr().round(2)

L'année et le kilométrage sont liés (−0,83) : une voiture récente a peu roulé. Le nuage de
points le montre. Chaque point est une voiture.

In [ ]:
ventes.plot(kind="scatter", x="annee", y="kilometrage", figsize=(8, 4));

**Deux colonnes jumelles.** Ajoutons une colonne `age`, l'âge de la voiture en 2025. Elle ne dit
rien de plus que l'année :

In [ ]:
ventes["age"] = 2025 - ventes["annee"]
ventes[["annee", "age"]].corr()

r = −1 : la même information, dans l'autre sens. On n'en garde qu'une.

In [ ]:
ventes = ventes.drop(columns=["age"])
list(ventes.columns)

## 9. Le texte que la machine ne sait pas lire

### L'état a un ordre : encodage ordinal

Mauvais < Correct < Bon < Très bon. Des nombres qui respectent cet ordre disent la vérité.

In [ ]:
ordre_etat = {"Mauvais": 0, "Correct": 1, "Bon": 2, "Très bon": 3}
ventes["etat_code"] = ventes["etat"].map(ordre_etat)

ventes.groupby("etat_code")["prix"].median()

Plus l'état est bon, plus le prix médian monte : l'ordre porte une information, la machine
pourra s'en servir.

### Le carburant n'a pas d'ordre : encodage one-hot

Une colonne par carburant, avec 1 ou 0. Aucune fausse hiérarchie.

In [ ]:
pd.get_dummies(ventes["carburant"], prefix="carburant", dtype=int).head(8)

## 10. La même échelle pour tous

**Normalisation** : ramener entre 0 (le plus petit) et 1 (le plus grand).

**Standardisation** : moyenne 0, écart-type 1. Chaque valeur devient son z-score.

In [ ]:
km = ventes["kilometrage"]
ventes["km_normalise"] = (km - km.min()) / (km.max() - km.min())
ventes["km_standardise"] = (km - km.mean()) / km.std(ddof=0)

ventes[["kilometrage", "km_normalise", "km_standardise"]].describe().round(2)

Après normalisation : minimum 0, maximum 1. Après standardisation : moyenne 0, écart-type 1,
et pas de bornes.

**Les mêmes gestes existent dans scikit-learn**, la bibliothèque de machine learning que nous
utiliserons dans les prochaines séances :

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

colonnes = ["annee", "kilometrage", "remise"]
normalise = MinMaxScaler().fit_transform(ventes[colonnes])
standardise = StandardScaler().fit_transform(ventes[colonnes])

print("normalisé, 3 premières lignes :")
print(normalise[:3].round(2))
print("standardisé, 3 premières lignes :")
print(standardise[:3].round(2))

## Enregistrer le fichier propre

In [ ]:
ventes.to_excel("donnees/ventes-propres.xlsx", index=False)
print("Fichier propre enregistré : donnees/ventes-propres.xlsx")